# Etapa 1 — Baselines: DummyClassifier e Regressão Logística

Treinamento de modelos baseline para previsão de churn em telecomunicações.  
Todos os experimentos são rastreados no **MLflow** com validação cruzada estratificada (5-fold).

**Modelos:** DummyClassifier (most_frequent, stratified) e Regressão Logística  
**Métricas monitoradas:** Accuracy, AUC-ROC, PR-AUC, F1, F1-weighted


In [1]:
import logging
import os
import sys
import warnings

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')
from preprocessing import RANDOM_STATE, build_preprocessor, load_and_clean_data, prepare_features

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
)
logger = logging.getLogger('baselines')
logger.info('Bibliotecas carregadas com sucesso.')

2026-04-21 17:48:14,724 - baselines - INFO - Bibliotecas carregadas com sucesso.


In [2]:
CV_FOLDS = 5
EXPERIMENT_NAME = 'telco-churn-baselines'

SCORING = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'f1': 'f1',
    'f1_weighted': 'f1_weighted',
}

logger.info(
    'Configuracao: RANDOM_STATE=%d, CV_FOLDS=%d, EXPERIMENT=%s',
    RANDOM_STATE, CV_FOLDS, EXPERIMENT_NAME,
)

2026-04-21 17:48:14,730 - baselines - INFO - Configuracao: RANDOM_STATE=42, CV_FOLDS=5, EXPERIMENT=telco-churn-baselines


In [3]:
df = load_and_clean_data()
X, y = prepare_features(df)

logger.info(
    'Classe alvo - 0 (Sem Churn): %d (%.1f%%) | 1 (Churn): %d (%.1f%%)',
    (y == 0).sum(), 100 * (y == 0).mean(),
    (y == 1).sum(), 100 * (y == 1).mean(),
)
display(df.head(3))

2026-04-21 17:48:15,658 - preprocessing - INFO - Dataset carregado: 7043 linhas, 21 colunas. 11 valores imputados em TotalCharges.


2026-04-21 17:48:15,661 - preprocessing - INFO - Features: (7043, 19) | Distribuicao alvo: 0=5174 (73.5%) 1=1869 (26.5%)


2026-04-21 17:48:15,662 - baselines - INFO - Classe alvo - 0 (Sem Churn): 5174 (73.5%) | 1 (Churn): 1869 (26.5%)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,target
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1


In [4]:
# Aponta o MLflow para a raiz do projeto (funciona rodando de notebooks/ ou da raiz)
_parent = os.path.abspath(os.path.join('..', 'mlruns'))
_local  = os.path.abspath('mlruns')
_mlruns_path = _parent if os.path.exists(os.path.join('..', 'mlruns')) else _local
mlflow.set_tracking_uri('file:///' + _mlruns_path.replace('\\', '/'))

mlflow.set_experiment(EXPERIMENT_NAME)

dataset_info = mlflow.data.from_pandas(
    df,
    name='telco_customer_churn',
    targets='target',
)

logger.info('MLflow URI: %s | Experimento: %s', mlflow.get_tracking_uri(), EXPERIMENT_NAME)

2026-04-21 17:48:15,676 - root - WARNING - Malformed experiment '1'. Detailed error Yaml file 'C:\Users\ermir\Documents\Github\Tech-Challenge-Fase1\mlruns\1\meta.yaml' does not exist.
Traceback (most recent call last):
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           

2026/04/21 17:48:15 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-baselines' does not exist. Creating a new experiment.


2026-04-21 17:48:15,682 - root - WARNING - Malformed experiment '1'. Detailed error Yaml file 'C:\Users\ermir\Documents\Github\Tech-Challenge-Fase1\mlruns\1\meta.yaml' does not exist.
Traceback (most recent call last):
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ermir\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           

2026-04-21 17:48:15,803 - baselines - INFO - MLflow URI: file:///C:/Users/ermir/Documents/Github/Tech-Challenge-Fase1/mlruns | Experimento: telco-churn-baselines


In [5]:
def run_cv_and_log(model_name, clf_pipeline, extra_params):
    """Execute cross-validation estratificada e registra resultados no MLflow."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    logger.info('Iniciando CV para: %s', model_name)

    cv_results = cross_validate(clf_pipeline, X, y, cv=cv, scoring=SCORING)

    metrics = {}
    for key, values in cv_results.items():
        if key.startswith('test_'):
            name = key[5:]
            metrics[name] = float(values.mean())
            metrics[name + '_std'] = float(values.std())

    params = {
        'model_name': model_name,
        'cv_folds': CV_FOLDS,
        'random_state': RANDOM_STATE,
        'dataset_rows': len(X),
        'dataset_features': X.shape[1],
    }
    params.update(extra_params)

    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_input(dataset_info, context='training')
        mlflow.sklearn.log_model(clf_pipeline, artifact_path='model')
        run_id = run.info.run_id

    logger.info(
        '%s | AUC-ROC=%.4f (+/-%.4f) | F1=%.4f | Accuracy=%.4f',
        model_name,
        metrics['roc_auc'], metrics['roc_auc_std'],
        metrics['f1'], metrics['accuracy'],
    )
    return {'run_id': run_id, 'metrics': metrics, 'model_name': model_name}

## DummyClassifier Baseline

Treinamos duas estratégias do DummyClassifier como **limite inferior de desempenho**:
- **most_frequent**: sempre prediz a classe majoritária (Não Churn — ~73%)
- **stratified**: prediz aleatoriamente respeitando a distribuição das classes


In [6]:
results_list = []

for strategy in ['most_frequent', 'stratified']:
    dummy_clf = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE)
    clf_pipeline = Pipeline(steps=[
        ('preprocessor', build_preprocessor()),
        ('classifier', dummy_clf),
    ])

    result = run_cv_and_log(
        model_name='DummyClassifier_' + strategy,
        clf_pipeline=clf_pipeline,
        extra_params={'strategy': strategy},
    )
    results_list.append(result)

logger.info('%d runs DummyClassifier registradas no MLflow.', len(results_list))

2026-04-21 17:48:15,822 - preprocessing - INFO - Preprocessor configurado: 4 features numericas, 15 features categoricas.


2026-04-21 17:48:15,822 - baselines - INFO - Iniciando CV para: DummyClassifier_most_frequent


2026/04/21 17:48:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/21 17:48:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-04-21 17:48:19,488 - baselines - INFO - DummyClassifier_most_frequent | AUC-ROC=0.5000 (+/-0.0000) | F1=0.0000 | Accuracy=0.7346


...

2026-04-21 17:48:19,490 - baselines - INFO - Iniciando CV para: DummyClassifier_stratified


2026/04/21 17:48:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/21 17:48:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-04-21 17:48:22,219 - baselines - INFO - DummyClassifier_stratified | AUC-ROC=0.5050 (+/-0.0073) | F1=0.2738 | Accuracy=0.6129


2026-04-21 17:48:22,220 - baselines - INFO - 2 runs DummyClassifier registradas no MLflow.


In [7]:
summary_rows = []
for r in results_list:
    row = {'modelo': r['model_name']}
    for m in ['accuracy', 'roc_auc', 'average_precision', 'f1', 'f1_weighted']:
        row[m] = round(r['metrics'].get(m, 0.0), 4)
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows).set_index('modelo')
display(results_df)
logger.info('Resumo DummyClassifier exibido.')

,accuracy,roc_auc,average_precision,f1,f1_weighted
modelo,,,,,
DummyClassifier_most_frequent,0.7346,0.500,0.2654,0.0000,0.6222
DummyClassifier_stratified,0.6129,0.505,0.2675,0.2738,0.6135


2026-04-21 17:48:22,227 - baselines - INFO - Resumo DummyClassifier exibido.


## Regressão Logística Baseline

Treinamos Regressão Logística como baseline com capacidade de aprendizado real.  
Usamos `class_weight='balanced'` para lidar com o desbalanceamento (~73% sem churn vs ~27% com churn).

**Esperado:** superar significativamente o DummyClassifier, especialmente em AUC-ROC e F1.


In [8]:
lr_params = {
    'C': 1.0,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'class_weight': 'balanced',
}

lr_clf = LogisticRegression(random_state=RANDOM_STATE, **lr_params)
lr_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', lr_clf),
])

result_lr = run_cv_and_log(
    model_name='LogisticRegression',
    clf_pipeline=lr_pipeline,
    extra_params=lr_params,
)
results_list.append(result_lr)

logger.info('Regressao Logistica concluida e registrada no MLflow.')

2026-04-21 17:48:22,232 - preprocessing - INFO - Preprocessor configurado: 4 features numericas, 15 features categoricas.


2026-04-21 17:48:22,232 - baselines - INFO - Iniciando CV para: LogisticRegression


2026/04/21 17:48:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/21 17:48:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026-04-21 17:48:25,102 - baselines - INFO - LogisticRegression | AUC-ROC=0.8449 (+/-0.0134) | F1=0.6258 | Accuracy=0.7457


2026-04-21 17:48:25,104 - baselines - INFO - Regressao Logistica concluida e registrada no MLflow.


## Comparação de Modelos Baseline

Tabela comparativa com métricas de validação cruzada estratificada (média ± desvio padrão, 5-fold).


In [9]:
comparison_rows = []
for r in results_list:
    row = {'Modelo': r['model_name']}
    for metric in ['accuracy', 'roc_auc', 'average_precision', 'f1', 'f1_weighted']:
        mean_val = r['metrics'].get(metric, 0.0)
        std_val = r['metrics'].get(metric + '_std', 0.0)
        row[metric] = str(round(mean_val, 4)) + ' +/- ' + str(round(std_val, 4))
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Modelo')
display(comparison_df)

logger.info('Tabela comparativa exibida. Todos os experimentos registrados no MLflow.')

,accuracy,roc_auc,average_precision,f1,f1_weighted
Modelo,,,,,
DummyClassifier_most_frequent,0.7346 +/- 0.0002,0.5 +/- 0.0,0.2654 +/- 0.0002,0.0 +/- 0.0,0.6222 +/- 0.0003
DummyClassifier_stratified,0.6129 +/- 0.0056,0.505 +/- 0.0073,0.2675 +/- 0.0031,0.2738 +/- 0.0107,0.6135 +/- 0.0056
LogisticRegression,0.7457 +/- 0.0054,0.8449 +/- 0.0134,0.6554 +/- 0.0275,0.6258 +/- 0.0089,0.7592 +/- 0.0051


2026-04-21 17:48:25,111 - baselines - INFO - Tabela comparativa exibida. Todos os experimentos registrados no MLflow.


## Conclusão

Todos os experimentos foram registrados no MLflow com:
- **Parâmetros**: hiperparâmetros do modelo, configuração de CV (folds, random_state)
- **Métricas**: accuracy, AUC-ROC, PR-AUC, F1 e F1-weighted (média e desvio padrão)
- **Dataset**: versão e informações do conjunto de dados (7.043 registros, 19 features)
- **Artefatos**: modelo serializado via MLflow sklearn

A Regressão Logística supera significativamente os DummyClassifiers,
estabelecendo o **baseline de referência** para comparação com a MLP PyTorch na Etapa 2.
